セル1 ライブラリインストール

In [10]:
!pip -q install pybullet google-genai imageio imageio-ffmpeg pillow

セル2 初期化・Gemini・ログ

In [21]:
client = genai.Client(
    api_key=API_KEY
)

MODEL_NAME = "gemini-3.5-flash"

write_log(
    f"Gemini Model={MODEL_NAME}"
)

[08:48:08] Gemini Model=gemini-3.5-flash


In [22]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

接続確認コード

In [23]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="こんにちは"
)

print(response.text)

こんにちは！お手伝いできることはありますか？

質問や調べたいこと、話したいテーマなどがあれば、何でも気軽に教えてくださいね。


セル3 Geminiによる目標地点取得

In [24]:
MODEL_NAME = "gemini-3.5-flash"

def get_goal(user_instruction):

    write_log(
        f"Instruction={user_instruction}"
    )

    prompt = f"""
以下の文章から目的地の座標を抽出してください。

出力はJSONのみとしてください。

例：
{{"goal_x":3,"goal_y":0}}

入力:
{user_instruction}
"""

    try:

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        write_log(
            f"Model={MODEL_NAME}"
        )

        write_log(
            f"Gemini Response={response.text}"
        )

        match = re.search(
            r"\{.*\}",
            response.text,
            re.DOTALL
        )

        if not match:

            write_log(
                "JSON解析失敗。デフォルト値を使用"
            )

            return 3.0, 0.0

        data = json.loads(
            match.group()
        )

        goal_x = float(
            data["goal_x"]
        )

        goal_y = float(
            data["goal_y"]
        )

        write_log(
            f"Goal=({goal_x},{goal_y})"
        )

        return (
            goal_x,
            goal_y
        )

    except Exception as e:

        write_log(
            f"Gemini Error={e}"
        )

        write_log(
            "デフォルト座標を使用"
        )

        return 3.0, 0.0


goal_x, goal_y = get_goal(
    "障害物を避けて座標(3,0)へ移動してください"
)

print(
    f"Goal = ({goal_x}, {goal_y})"
)

[08:48:29] Instruction=障害物を避けて座標(3,0)へ移動してください
[08:48:30] Model=gemini-3.5-flash
[08:48:30] Gemini Response={"goal_x":3,"goal_y":0}
[08:48:30] Goal=(3.0,0.0)
Goal = (3.0, 0.0)


セル4（PyBullet・R2D2・障害物・ゴール）

In [102]:
import pybullet as p
import pybullet_data

write_log("PyBullet Start")

if "goal_x" not in globals():
    goal_x = 3.0

if "goal_y" not in globals():
    goal_y = 0.0

try:
    p.disconnect()
except:
    pass

p.connect(p.DIRECT)

p.resetSimulation()

p.setAdditionalSearchPath(
    pybullet_data.getDataPath()
)

p.setGravity(0,0,-9.8)

plane = p.loadURDF(
    "plane.urdf"
)

robot = p.loadURDF(
    "r2d2.urdf",
    [0,0,0.2]
)

# ゴール（緑色）

goal_marker = p.loadURDF(
    "sphere_small.urdf",
    [goal_x,goal_y,0.4]
)

p.changeVisualShape(
    goal_marker,
    -1,
    rgbaColor=[0,1,0,1]
)

# 赤色障害物を縦一列

OBSTACLES = [
    (1.5,-0.8),
    (1.5, 0.0),
    (1.5, 0.8)
]

for x,y in OBSTACLES:

    obstacle = p.loadURDF(
        "cube_small.urdf",
        [x,y,0.2],
        useFixedBase=True
    )

    p.changeVisualShape(
        obstacle,
        -1,
        rgbaColor=[1,0,0,1]
    )

robot_x = 0.0
robot_y = 0.0
robot_yaw = 0.0

STEP = 0.015

# 交互に左右を通るジグザグ経路

PATROL_ROUTE = [

    (1.0,-1.20),

    (2.1,-0.45),

    (0.9, 0.10),

    (2.1, 0.65),

    (0.9, 1.25),

    (2.7, 0.60)

]

print("Bodies =", p.getNumBodies())

write_log("Simulation Ready")

[11:16:00] PyBullet Start
Bodies = 6
[11:16:00] Simulation Ready


セル5（移動制御・カメラ）

In [103]:
from PIL import Image
from PIL import ImageDraw
import numpy as np
import math

def capture():

    pos,_ = p.getBasePositionAndOrientation(
        robot
    )

    view = p.computeViewMatrixFromYawPitchRoll(
        cameraTargetPosition=[
            pos[0],
            pos[1],
            0
        ],
        distance=3,
        yaw=20,
        pitch=-18,
        roll=0,
        upAxisIndex=2
    )

    projection = p.computeProjectionMatrixFOV(
        fov=75,
        aspect=640/480,
        nearVal=0.1,
        farVal=100
    )

    img = p.getCameraImage(
        width=640,
        height=480,
        viewMatrix=view,
        projectionMatrix=projection
    )

    rgb = np.array(
        img[2],
        dtype=np.uint8
    ).reshape((480,640,4))

    frame = rgb[:,:,:3]

    image = Image.fromarray(frame)

    draw = ImageDraw.Draw(image)

    draw.rectangle(
        (0,0,640,80),
        fill="white"
    )

    remaining = math.sqrt(
        (robot_x-goal_x)**2+
        (robot_y-goal_y)**2
    )

    draw.text(
        (10,10),
        "AI ROBOT MISSION",
        fill="black"
    )

    draw.text(
        (10,40),
        f"Robot=({robot_x:.2f},{robot_y:.2f})",
        fill="black"
    )

    draw.text(
        (230,10),
        f"Remain={remaining:.2f}",
        fill="blue"
    )

    draw.text(
        (230,40),
        f"Goal=({goal_x:.2f},{goal_y:.2f})",
        fill="black"
    )

    draw.text(
        (450,10),
        "RED=Obstacle",
        fill="red"
    )

    draw.text(
        (450,40),
        "GREEN=Goal",
        fill="green"
    )

    return np.array(image)


def move_to_target(
        target_x,
        target_y,
        frames):

    global robot_x
    global robot_y
    global robot_yaw

    count = 0

    while count < 3000:

        dx = target_x - robot_x
        dy = target_y - robot_y

        distance = math.sqrt(
            dx*dx+dy*dy
        )

        if distance < 0.05:
            break

        target_yaw = math.atan2(
            dy,
            dx
        )

        yaw_diff = target_yaw - robot_yaw

        while yaw_diff > math.pi:
            yaw_diff -= 2*math.pi

        while yaw_diff < -math.pi:
            yaw_diff += 2*math.pi

        robot_yaw += yaw_diff * 0.1

        robot_x += (
            STEP *
            math.cos(robot_yaw)
        )

        robot_y += (
            STEP *
            math.sin(robot_yaw)
        )

        quat = p.getQuaternionFromEuler(
            [0,0,robot_yaw]
        )

        p.resetBasePositionAndOrientation(
            robot,
            [robot_x,
             robot_y,
             0.2],
            quat
        )

        p.stepSimulation()

        frames.append(
            capture()
        )

        count += 1

セル6（動画生成）

In [105]:
import imageio.v2 as imageio

frames = []

FPS = 10

OPENING_FRAMES = 30     # 3秒
MOVE_FRAMES = 1200      # 120秒
ENDING_FRAMES = 50      # 5秒

# OPENING

opening = Image.new(
    "RGB",
    (640,480),
    "black"
)

draw = ImageDraw.Draw(opening)

draw.text(
    (80,120),
    "AI ROBOT MISSION",
    fill="white"
)

draw.text(
    (80,200),
    "R2D2 Zigzag Navigation",
    fill="yellow"
)

for _ in range(OPENING_FRAMES):
    frames.append(
        np.array(opening)
    )

# 本編2分

move_start = len(frames)

while (
    len(frames) - move_start
) < MOVE_FRAMES:

    for tx,ty in PATROL_ROUTE:

        move_to_target(
            tx,
            ty,
            frames
        )

        if (
            len(frames)-move_start
        ) >= MOVE_FRAMES:
            break

# 最後にゴール

move_to_target(
    goal_x,
    goal_y,
    frames
)

for _ in range(100):
    frames.append(
        capture()
    )

distance = math.sqrt(
    (robot_x-goal_x)**2+
    (robot_y-goal_y)**2
)

success = distance < 0.2

# ENDING

ending = Image.new(
    "RGB",
    (640,480),
    "black"
)

draw = ImageDraw.Draw(ending)

draw.text(
    (80,120),
    "MISSION COMPLETE",
    fill="lime"
)

draw.text(
    (80,200),
    f"Success={success}",
    fill="white"
)

draw.text(
    (80,260),
    f"Distance={distance:.2f}",
    fill="white"
)

for _ in range(ENDING_FRAMES):
    frames.append(
        np.array(ending)
    )

VIDEO_FILE = "ai_robot_demo.mp4"

imageio.mimsave(
    VIDEO_FILE,
    [
        f.astype(np.uint8)
        for f in frames
        if isinstance(f,np.ndarray)
        and f.ndim == 3
        and f.shape[2] == 3
    ],
    fps=FPS
)

from IPython.display import Video

Video(
    VIDEO_FILE,
    embed=True
)